In [1]:
import pandas as pd

df = pd.read_csv("tennis.csv")

print(df.head())
print(df.columns)

    Outlook Temperature Humidity   Windy PlayTennis
0     Sunny         Hot     High    Weak         No
1     Sunny         Hot     High  Strong         No
2  Overcast         Hot     High    Weak        Yes
3     Rainy        Mild     High    Weak        Yes
4     Rainy        Cool   Normal    Weak        Yes
Index(['Outlook', 'Temperature', 'Humidity', 'Windy', 'PlayTennis'], dtype='str')


In [2]:
# Select the attributes and target column
attributes = df.columns[:-1]
target = df.columns[-1]

print("Attributes:", list(attributes))
print("Target:", target)

Attributes: ['Outlook', 'Temperature', 'Humidity', 'Windy']
Target: PlayTennis


In [3]:
# Initialize the hypothesis
hypothesis = None

print("Initial Hypothesis:", hypothesis)

Initial Hypothesis: None


In [4]:

# FIND-S Algorithm using Tennis Dataset

import pandas as pd

# Load the dataset (if not already loaded)
df = pd.read_csv("tennis.csv")

# Initialize hypothesis
hypothesis = None

# Process each training example
for i in range(len(df)):
    example = list(df.iloc[i, :-1])
    target = df.iloc[i, -1]

    print("Example:", example, "| PlayTennis:", target)

    # Consider only positive examples
    if target == "Yes":
        if hypothesis is None:
            hypothesis = example.copy()
        else:
            for j in range(len(hypothesis)):
                if hypothesis[j] != example[j]:
                    hypothesis[j] = "?"

        print("Current Hypothesis:", hypothesis)

print("\nFinal Hypothesis:", hypothesis)


Example: ['Sunny', 'Hot', 'High', 'Weak'] | PlayTennis: No
Example: ['Sunny', 'Hot', 'High', 'Strong'] | PlayTennis: No
Example: ['Overcast', 'Hot', 'High', 'Weak'] | PlayTennis: Yes
Current Hypothesis: ['Overcast', 'Hot', 'High', 'Weak']
Example: ['Rainy', 'Mild', 'High', 'Weak'] | PlayTennis: Yes
Current Hypothesis: ['?', '?', 'High', 'Weak']
Example: ['Rainy', 'Cool', 'Normal', 'Weak'] | PlayTennis: Yes
Current Hypothesis: ['?', '?', '?', 'Weak']
Example: ['Rainy', 'Cool', 'Normal', 'Strong'] | PlayTennis: No
Example: ['Overcast', 'Cool', 'Normal', 'Strong'] | PlayTennis: Yes
Current Hypothesis: ['?', '?', '?', '?']
Example: ['Sunny', 'Mild', 'High', 'Weak'] | PlayTennis: No
Example: ['Sunny', 'Cool', 'Normal', 'Weak'] | PlayTennis: Yes
Current Hypothesis: ['?', '?', '?', '?']
Example: ['Rainy', 'Mild', 'Normal', 'Weak'] | PlayTennis: Yes
Current Hypothesis: ['?', '?', '?', '?']
Example: ['Sunny', 'Mild', 'Normal', 'Strong'] | PlayTennis: Yes
Current Hypothesis: ['?', '?', '?', '?']

In [5]:

import pandas as pd

# Load the dataset
df = pd.read_csv("tennis.csv")

attributes = list(df.columns[:-1])
target = df.columns[-1]

# Possible values of each attribute
domains = [df[col].unique().tolist() for col in attributes]

# Check whether h1 is more general than or equal to h2
def more_general(h1, h2):
    return all(a == "?" or a == b for a, b in zip(h1, h2))

# Check whether a hypothesis covers an example
def covers(example, hypothesis):
    return all(h == "?" or h == x
               for x, h in zip(example, hypothesis))

# Remove duplicate hypotheses
def remove_duplicates(hypotheses):
    result = []
    for h in hypotheses:
        if h not in result:
            result.append(h)
    return result

# Initialize specific and general boundaries
S = [["0"] * len(attributes)]
G = [["?"] * len(attributes)]

print("Initial S:", S)
print("Initial G:", G)

# Process training examples
for index, row in df.iterrows():
    example = row[attributes].tolist()
    label = row[target]

    print("\nExample", index + 1, ":", example, "=>", label)

    if label == "Yes":
        # Remove general hypotheses that do not cover a positive example
        G = [g for g in G if covers(example, g)]

        new_S = []
        for s in S:
            if all(x == "0" for x in s):
                new_S.append(example.copy())
            elif covers(example, s):
                new_S.append(s)
            else:
                h = [
                    s[i] if s[i] == example[i] else "?"
                    for i in range(len(attributes))
                ]
                new_S.append(h)

        # Keep only hypotheses bounded by G
        S = [
            s for s in remove_duplicates(new_S)
            if any(more_general(g, s) for g in G)
        ]

    else:
        # Remove specific hypotheses that cover a negative example
        S = [s for s in S if not covers(example, s)]

        new_G = []
        for g in G:
            if not covers(example, g):
                new_G.append(g)
            else:
                # Specialize G to exclude this negative example
                for i in range(len(attributes)):
                    if g[i] == "?":
                        for value in domains[i]:
                            if value != example[i]:
                                h = g.copy()
                                h[i] = value

                                if any(
                                    more_general(h, s) for s in S
                                ):
                                    new_G.append(h)

        # Keep maximally general hypotheses
        new_G = remove_duplicates(new_G)
        G = [
            g for g in new_G
            if not any(
                g != other and more_general(other, g)
                for other in new_G
            )
        ]

    print("S:", S)
    print("G:", G)

print("\nFINAL SPECIFIC BOUNDARY (S):")
for h in S:
    print(h)

print("\nFINAL GENERAL BOUNDARY (G):")
for h in G:
    print(h)


Initial S: [['0', '0', '0', '0']]
Initial G: [['?', '?', '?', '?']]

Example 1 : ['Sunny', 'Hot', 'High', 'Weak'] => No
S: [['0', '0', '0', '0']]
G: []

Example 2 : ['Sunny', 'Hot', 'High', 'Strong'] => No
S: [['0', '0', '0', '0']]
G: []

Example 3 : ['Overcast', 'Hot', 'High', 'Weak'] => Yes
S: []
G: []

Example 4 : ['Rainy', 'Mild', 'High', 'Weak'] => Yes
S: []
G: []

Example 5 : ['Rainy', 'Cool', 'Normal', 'Weak'] => Yes
S: []
G: []

Example 6 : ['Rainy', 'Cool', 'Normal', 'Strong'] => No
S: []
G: []

Example 7 : ['Overcast', 'Cool', 'Normal', 'Strong'] => Yes
S: []
G: []

Example 8 : ['Sunny', 'Mild', 'High', 'Weak'] => No
S: []
G: []

Example 9 : ['Sunny', 'Cool', 'Normal', 'Weak'] => Yes
S: []
G: []

Example 10 : ['Rainy', 'Mild', 'Normal', 'Weak'] => Yes
S: []
G: []

Example 11 : ['Sunny', 'Mild', 'Normal', 'Strong'] => Yes
S: []
G: []

Example 12 : ['Overcast', 'Mild', 'High', 'Strong'] => Yes
S: []
G: []

Example 13 : ['Overcast', 'Hot', 'Normal', 'Weak'] => Yes
S: []
G: []

E

In [6]:

import pandas as pd

df = pd.read_csv("tennis.csv")
attributes = list(df.columns[:-1])
target = df.columns[-1]
domains = [df[col].unique().tolist() for col in attributes]

def covers(example, h):
    return all(h[i] == "?" or h[i] == example[i]
               for i in range(len(h)))

def more_general_or_equal(h1, h2):
    return all(a == "?" or a == b
               for a, b in zip(h1, h2))

def remove_duplicates(hypotheses):
    result = []
    for h in hypotheses:
        if h not in result:
            result.append(h)
    return result

def minimal_generalizations(s, example):
    return [[example[i] if s[i] == "0" else
             (s[i] if s[i] == example[i] else "?")
             for i in range(len(s))]]

def minimal_specializations(g, example, S):
    result = []
    for i in range(len(g)):
        if g[i] == "?":
            for value in domains[i]:
                if value != example[i]:
                    h = g.copy()
                    h[i] = value
                    if any(more_general_or_equal(h, s) for s in S):
                        result.append(h)
    return result

S = [["0"] * len(attributes)]
G = [["?"] * len(attributes)]

for _, row in df.iterrows():
    example = row[attributes].tolist()
    label = row[target]

    if label == "Yes":
        G = [g for g in G if covers(example, g)]
        new_S = []
        for s in S:
            if covers(example, s):
                new_S.append(s)
            else:
                new_S.extend(minimal_generalizations(s, example))
        S = remove_duplicates([
            s for s in new_S
            if any(more_general_or_equal(g, s) for g in G)
        ])

    else:
        S = [s for s in S if not covers(example, s)]
        new_G = []
        for g in G:
            if covers(example, g):
                new_G.extend(minimal_specializations(g, example, S))
            else:
                new_G.append(g)
        G = remove_duplicates([
            g for g in new_G
            if any(more_general_or_equal(g, s) for s in S)
        ])

    print("Example:", example, "=>", label)
    print("S:", S)
    print("G:", G)
    print()

print("FINAL SPECIFIC BOUNDARY:", S)
print("FINAL GENERAL BOUNDARY:", G)


Example: ['Sunny', 'Hot', 'High', 'Weak'] => No
S: [['0', '0', '0', '0']]
G: []

Example: ['Sunny', 'Hot', 'High', 'Strong'] => No
S: [['0', '0', '0', '0']]
G: []

Example: ['Overcast', 'Hot', 'High', 'Weak'] => Yes
S: []
G: []

Example: ['Rainy', 'Mild', 'High', 'Weak'] => Yes
S: []
G: []

Example: ['Rainy', 'Cool', 'Normal', 'Weak'] => Yes
S: []
G: []

Example: ['Rainy', 'Cool', 'Normal', 'Strong'] => No
S: []
G: []

Example: ['Overcast', 'Cool', 'Normal', 'Strong'] => Yes
S: []
G: []

Example: ['Sunny', 'Mild', 'High', 'Weak'] => No
S: []
G: []

Example: ['Sunny', 'Cool', 'Normal', 'Weak'] => Yes
S: []
G: []

Example: ['Rainy', 'Mild', 'Normal', 'Weak'] => Yes
S: []
G: []

Example: ['Sunny', 'Mild', 'Normal', 'Strong'] => Yes
S: []
G: []

Example: ['Overcast', 'Mild', 'High', 'Strong'] => Yes
S: []
G: []

Example: ['Overcast', 'Hot', 'Normal', 'Weak'] => Yes
S: []
G: []

Example: ['Rainy', 'Mild', 'High', 'Strong'] => No
S: []
G: []

FINAL SPECIFIC BOUNDARY: []
FINAL GENERAL BOUNDAR

In [7]:

import pandas as pd

df = pd.read_csv("tennis.csv")

attributes = list(df.columns[:-1])
target = df.columns[-1]
domains = [df[col].unique().tolist() for col in attributes]

def covers(example, h):
    return all(h[i] == "?" or h[i] == example[i]
               for i in range(len(h)))

def more_general_or_equal(h1, h2):
    return all(a == "?" or a == b
               for a, b in zip(h1, h2))

def remove_duplicates(hypotheses):
    result = []
    for h in hypotheses:
        if h not in result:
            result.append(h)
    return result

def minimal_specializations(g, example, S):
    result = []
    for i in range(len(g)):
        if g[i] == "?":
            for value in domains[i]:
                if value != example[i]:
                    h = g.copy()
                    h[i] = value
                    if any(more_general_or_equal(h, s) for s in S):
                        result.append(h)
    return result

S = [["0"] * len(attributes)]
G = [["?"] * len(attributes)]

for _, row in df.iterrows():
    example = row[attributes].tolist()
    label = row[target]

    if label == "Yes":
        G = [g for g in G if covers(example, g)]

        new_S = []
        for s in S:
            if all(v == "0" for v in s):
                new_S.append(example.copy())
            elif covers(example, s):
                new_S.append(s)
            else:
                new_S.append([
                    s[i] if s[i] == example[i] else "?"
                    for i in range(len(s))
                ])

        S = remove_duplicates([
            s for s in new_S
            if any(more_general_or_equal(g, s) for g in G)
        ])

    else:
        S = [s for s in S if not covers(example, s)]
        new_G = []

        for g in G:
            if covers(example, g):
                new_G.extend(minimal_specializations(g, example, S))
            else:
                new_G.append(g)

        G = remove_duplicates([
            g for g in new_G
            if any(more_general_or_equal(g, s) for s in S)
        ])

    print(f"{example} => {label}")
    print("S =", S)
    print("G =", G)
    print()

print("FINAL S:", S)
print("FINAL G:", G)


['Sunny', 'Hot', 'High', 'Weak'] => No
S = [['0', '0', '0', '0']]
G = []

['Sunny', 'Hot', 'High', 'Strong'] => No
S = [['0', '0', '0', '0']]
G = []

['Overcast', 'Hot', 'High', 'Weak'] => Yes
S = []
G = []

['Rainy', 'Mild', 'High', 'Weak'] => Yes
S = []
G = []

['Rainy', 'Cool', 'Normal', 'Weak'] => Yes
S = []
G = []

['Rainy', 'Cool', 'Normal', 'Strong'] => No
S = []
G = []

['Overcast', 'Cool', 'Normal', 'Strong'] => Yes
S = []
G = []

['Sunny', 'Mild', 'High', 'Weak'] => No
S = []
G = []

['Sunny', 'Cool', 'Normal', 'Weak'] => Yes
S = []
G = []

['Rainy', 'Mild', 'Normal', 'Weak'] => Yes
S = []
G = []

['Sunny', 'Mild', 'Normal', 'Strong'] => Yes
S = []
G = []

['Overcast', 'Mild', 'High', 'Strong'] => Yes
S = []
G = []

['Overcast', 'Hot', 'Normal', 'Weak'] => Yes
S = []
G = []

['Rainy', 'Mild', 'High', 'Strong'] => No
S = []
G = []

FINAL S: []
FINAL G: []


In [8]:

import pandas as pd

df = pd.read_csv("tennis.csv")

attributes = list(df.columns[:-1])
target = df.columns[-1]
domains = [df[col].unique().tolist() for col in attributes]

def covers(example, h):
    return all(h[i] == "?" or h[i] == example[i]
               for i in range(len(h)))

def more_general_or_equal(h1, h2):
    return all(a == "?" or a == b
               for a, b in zip(h1, h2))

def remove_duplicates(hypotheses):
    result = []
    for h in hypotheses:
        if h not in result:
            result.append(h)
    return result

def minimal_specializations(g, example, S):
    result = []
    for i in range(len(g)):
        if g[i] == "?":
            for value in domains[i]:
                if value != example[i]:
                    h = g.copy()
                    h[i] = value
                    if any(more_general_or_equal(h, s) for s in S):
                        result.append(h)
    return result

S = [["0"] * len(attributes)]
G = [["?"] * len(attributes)]

for _, row in df.iterrows():
    example = row[attributes].tolist()
    label = row[target]

    if label == "Yes":
        G = [g for g in G if covers(example, g)]

        new_S = []
        for s in S:
            if all(v == "0" for v in s):
                new_S.append(example.copy())
            elif covers(example, s):
                new_S.append(s)
            else:
                new_S.append([
                    s[i] if s[i] == example[i] else "?"
                    for i in range(len(s))
                ])

        S = remove_duplicates([
            s for s in new_S
            if any(more_general_or_equal(g, s) for g in G)
        ])

    else:
        S = [s for s in S if not covers(example, s)]
        new_G = []

        for g in G:
            if covers(example, g):
                new_G.extend(minimal_specializations(g, example, S))
            else:
                new_G.append(g)

        G = remove_duplicates([
            g for g in new_G
            if any(more_general_or_equal(g, s) for s in S)
        ])

    print(f"{example} => {label}")
    print("S =", S)
    print("G =", G)
    print()

print("FINAL S:", S)
print("FINAL G:", G)


['Sunny', 'Hot', 'High', 'Weak'] => No
S = [['0', '0', '0', '0']]
G = []

['Sunny', 'Hot', 'High', 'Strong'] => No
S = [['0', '0', '0', '0']]
G = []

['Overcast', 'Hot', 'High', 'Weak'] => Yes
S = []
G = []

['Rainy', 'Mild', 'High', 'Weak'] => Yes
S = []
G = []

['Rainy', 'Cool', 'Normal', 'Weak'] => Yes
S = []
G = []

['Rainy', 'Cool', 'Normal', 'Strong'] => No
S = []
G = []

['Overcast', 'Cool', 'Normal', 'Strong'] => Yes
S = []
G = []

['Sunny', 'Mild', 'High', 'Weak'] => No
S = []
G = []

['Sunny', 'Cool', 'Normal', 'Weak'] => Yes
S = []
G = []

['Rainy', 'Mild', 'Normal', 'Weak'] => Yes
S = []
G = []

['Sunny', 'Mild', 'Normal', 'Strong'] => Yes
S = []
G = []

['Overcast', 'Mild', 'High', 'Strong'] => Yes
S = []
G = []

['Overcast', 'Hot', 'Normal', 'Weak'] => Yes
S = []
G = []

['Rainy', 'Mild', 'High', 'Strong'] => No
S = []
G = []

FINAL S: []
FINAL G: []
